# Experiment 01 — Data Audit

## 1. Introduction

This notebook examines the benchmark dataset used in the paper
"A Benchmark to Understand the Role of Knowledge Graphs on
Large Language Model's Accuracy for Question Answering on
Enterprise SQL Databases".

The dataset includes:
- Insurance data in CSV format
- SQL database schema (DDL)
- OWL ontology
- R2RML mappings
- Natural language questions and reference SQL/SPARQL queries

The purpose of this notebook is to verify the dataset before
reproducing the experiments.

In [2]:
from pathlib import Path

# Thư mục gốc của dự án
PROJECT_DIR = Path.cwd().parent

# Thư mục chứa dữ liệu benchmark
DATA_DIR = PROJECT_DIR / "data" / "ACME_Insurance"

# Các thành phần của benchmark
CSV_DIR = DATA_DIR / "data"
DDL_FILE = DATA_DIR / "DDL" / "ACME_small.ddl"
ONTOLOGY_FILE = DATA_DIR / "ontology" / "insurance.ttl"
MAPPING_FILE = CSV_DIR / "data.world_P&C_Insurance_Ontology_V1.r2rml"
BENCHMARK_FILE = DATA_DIR / "investigation" / "acme-benchmark.ttl"

print("Project:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

Project: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Dataset: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\data\ACME_Insurance
Dataset exists: True


In [3]:
# 1. Đếm số lượng file CSV
csv_files = sorted(CSV_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

# 2. Kiểm tra các file quan trọng
print("\nBenchmark components:")
print("SQL DDL:", DDL_FILE.exists())
print("OWL Ontology:", ONTOLOGY_FILE.exists())
print("R2RML Mapping:", MAPPING_FILE.exists())
print("Questions & References:", BENCHMARK_FILE.exists())

# 3. Liệt kê tên các file CSV
print("\nCSV files:")
for file in csv_files:
    print("-", file.name)

Number of CSV files: 29

Benchmark components:
SQL DDL: True
OWL Ontology: True
R2RML Mapping: True
Questions & References: True

CSV files:
- Agreement.csv
- Agreement_Assessment.csv
- Agreement_Party_Role.csv
- Assesment_Party_Role.csv
- Assessment.csv
- Assessment_Result.csv
- Catastrophe.csv
- Claim.csv
- Claim_Amount.csv
- Claim_Coverage.csv
- Expense_Payment.csv
- Expense_Reserve.csv
- FireClaim.csv
- Geographic_Location.csv
- Insurable_Object.csv
- Location_Address.csv
- Loss_Payment.csv
- Loss_Reserve.csv
- Occurrence.csv
- Organization.csv
- Party.csv
- Party_Role.csv
- Person.csv
- Physical_Location.csv
- Policy.csv
- Policy_Amount.csv
- Policy_Coverage_Detail.csv
- Premium.csv
- Underwriting_Assessment.csv


In [4]:
import pandas as pd

# Danh sách lưu thông tin các bảng
summary = []

# Đọc lần lượt từng file CSV
for file in csv_files:
    df = pd.read_csv(file)

    summary.append({
        "Table": file.stem,
        "Rows": len(df),
        "Columns": len(df.columns)
    })

# Hiển thị bảng tổng hợp
summary_df = pd.DataFrame(summary)
display(summary_df)

,Table,Rows,Columns
0,Agreement,2,5
1,Agreement_Assessment,2,2
2,Agreement_Party_Role,4,5
3,Assesment_Party_Role,2,5
4,Assessment,2,5
5,Assessment_Result,2,3
6,Catastrophe,4,5
7,Claim,2,14
8,Claim_Amount,8,7
9,Claim_Coverage,2,3


In [5]:
# Danh sách lưu kết quả
missing_summary = []

# Kiểm tra từng bảng CSV
for file in csv_files:
    df = pd.read_csv(file)

    missing_summary.append({
        "Table": file.stem,
        "Missing Values": df.isna().sum().sum(),
        "Columns with Missing": df.isna().any().sum()
    })

# Hiển thị kết quả
missing_df = pd.DataFrame(missing_summary)
display(missing_df)

,Table,Missing Values,Columns with Missing
0,Agreement,6,3
1,Agreement_Assessment,0,0
2,Agreement_Party_Role,8,2
3,Assesment_Party_Role,4,2
4,Assessment,4,2
5,Assessment_Result,0,0
6,Catastrophe,12,3
7,Claim,14,7
8,Claim_Amount,32,4
9,Claim_Coverage,0,0


In [6]:
# 1. Đọc dữ liệu bảng Claim
claim_df = pd.read_csv(CSV_DIR / "Claim.csv")

# 2. Kiểm tra giá trị thiếu theo từng cột
print("Missing values in Claim:")
print(claim_df.isna().sum())

# 3. Đọc SQL DDL
ddl_text = DDL_FILE.read_text(encoding="utf-8")

# 4. Trích xuất schema của bảng Claim
claim_schema = ddl_text.split("CREATE TABLE Claim", 1)[1]
claim_schema = claim_schema.split("CREATE TABLE Claim_Amount", 1)[0]

print("\nSQL Schema of Claim:")
print("CREATE TABLE Claim" + claim_schema)

Missing values in Claim:
Claim_Identifier                       0
Catastrophe_Identifier                 0
Claim_Description                      2
Claims_Made_Date                       2
Company_Claim_Number                   0
Company_Subclaim_Number                2
Insurable_Object_Identifier            0
Occurrence_Identifier                  0
Entry_Into_Claims_Made_Program_Date    2
Claim_Open_Date                        0
Claim_Close_Date                       0
Claim_Reopen_Date                      2
Claim_Status_Code                      2
Claim_Reported_Date                    2
dtype: int64

SQL Schema of Claim:
CREATE TABLE Claim
( 
	Claim_Identifier     int  NOT NULL ,
	Catastrophe_Identifier int  NULL ,
	Claim_Description    varchar(5000)  NULL ,
	Claims_Made_Date     datetime  NULL ,
	Company_Claim_Number varchar(20)  NULL ,
	Company_Subclaim_Number varchar(5)  NULL ,
	Insurable_Object_Identifier int  NULL ,
	Occurrence_Identifier int  NULL ,
	Entry_Into_Claims_Made_P

In [7]:
import re

pk_results = []

# Tìm từng định nghĩa CREATE TABLE trong DDL
tables = re.findall(
    r"CREATE TABLE\s+(\w+)\s*\((.*?)(?=\bCREATE TABLE\b|\Z)",
    ddl_text,
    flags=re.IGNORECASE | re.DOTALL
)

for table_name, table_ddl in tables:
    # Tìm định nghĩa PRIMARY KEY
    match = re.search(
        r"PRIMARY KEY\s*\(([^)]+)\)",
        table_ddl,
        flags=re.IGNORECASE
    )

    if not match:
        continue

    pk_columns = [
        col.strip().split()[0]
        for col in match.group(1).split(",")
    ]

    csv_path = CSV_DIR / f"{table_name}.csv"

    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path)

    # Đối chiếu tên cột, không phân biệt chữ hoa/thường
    column_lookup = {col.lower(): col for col in df.columns}

    if not all(col.lower() in column_lookup for col in pk_columns):
        continue

    actual_columns = [column_lookup[col.lower()] for col in pk_columns]

    pk_results.append({
        "Table": table_name,
        "Primary Key": ", ".join(pk_columns),
        "Missing": int(df[actual_columns].isna().any(axis=1).sum()),
        "Duplicates": int(df.duplicated(subset=actual_columns).sum())
    })

pk_df = pd.DataFrame(pk_results)
display(pk_df)

,Table,Primary Key,Missing,Duplicates
0,Claim,Claim_Identifier,0,0
1,Claim_Amount,Claim_Amount_Identifier,0,0
2,Loss_Payment,Claim_Amount_Identifier,0,0
3,Loss_Reserve,Claim_Amount_Identifier,0,0
4,Expense_Payment,Claim_Amount_Identifier,0,0
5,Expense_Reserve,Claim_Amount_Identifier,0,0
6,Claim_Coverage,"Claim_Identifier, Effective_Date, Policy_Cover...",0,0
7,Policy_Coverage_Detail,"Effective_Date, Policy_Coverage_Detail_Identifier",0,0
8,Policy,Policy_Identifier,0,0
9,Policy_Amount,Policy_Amount_Identifier,0,0


In [8]:
df = pd.read_csv(CSV_DIR / "Agreement_Party_Role.csv")

display(df)

print("Missing Effective_Date:", df["Effective_Date"].isna().sum())
print("Total rows:", len(df))

,Agreement_Identifier,Party_Identifier,Party_Role_Code,Effective_Date,Expiration_Date
0,1,1,PH,NaN,NaN
1,1,2,AG,NaN,NaN
2,2,1,PH,NaN,NaN
3,2,2,AG,NaN,NaN


Missing Effective_Date: 4
Total rows: 4


In [9]:
# Ánh xạ tên bảng với file CSV
csv_paths = {file.stem.lower(): file for file in csv_files}

fk_results = []

# Duyệt tất cả bảng trong DDL
for table_name, table_ddl in tables:

    child = pd.read_csv(csv_paths[table_name.lower()], dtype=str)

    # Tìm tất cả FOREIGN KEY
    foreign_keys = re.findall(
        r"FOREIGN KEY\s*\(([^)]+)\)\s*REFERENCES\s+(\w+)\s*\(([^)]+)\)",
        table_ddl,
        flags=re.IGNORECASE
    )

    for fk_text, parent_name, pk_text in foreign_keys:
        fk_cols = [c.strip() for c in fk_text.split(",")]
        pk_cols = [c.strip() for c in pk_text.split(",")]

        # Kiểm tra bảng được tham chiếu
        parent_path = csv_paths.get(parent_name.lower())

        if parent_path is None:
            status = "NOT CHECKED"
            invalid = None
        else:
            parent = pd.read_csv(parent_path, dtype=str)

            # Đối chiếu tên cột không phân biệt hoa/thường
            child_lookup = {c.lower(): c for c in child.columns}
            parent_lookup = {c.lower(): c for c in parent.columns}

            if (len(fk_cols) != len(pk_cols)
                or any(c.lower() not in child_lookup for c in fk_cols)
                or any(c.lower() not in parent_lookup for c in pk_cols)):
                status = "COLUMN MISMATCH"
                invalid = None
            else:
                fk_actual = [child_lookup[c.lower()] for c in fk_cols]
                pk_actual = [parent_lookup[c.lower()] for c in pk_cols]

                # Tạo tập các khóa ở bảng đích
                valid_keys = set(
                    parent[pk_actual].dropna().itertuples(
                        index=False, name=None
                    )
                )

                # Đếm các khóa ngoại không tìm thấy bảng đích
                invalid = sum(
                    row not in valid_keys
                    for row in child[fk_actual].dropna().itertuples(
                        index=False, name=None
                    )
                )

                status = "OK" if invalid == 0 else "INVALID"

        fk_results.append({
            "Table": table_name,
            "Foreign Key": ", ".join(fk_cols),
            "References": parent_name,
            "Invalid Rows": invalid,
            "Status": status
        })

fk_df = pd.DataFrame(fk_results)
display(fk_df)

,Table,Foreign Key,References,Invalid Rows,Status
0,Claim,Catastrophe_Identifier,Catastrophe,0.0,OK
1,Claim,Claim_Identifier,Claim,0.0,OK
2,Claim,Insurable_Object_Identifier,Insurable_Object,0.0,OK
3,Claim,Occurrence_Identifier,Occurrence,0.0,OK
4,Claim_Amount,Claim_Offer_Identifier,Claim_Offer,NaN,NOT CHECKED
5,Claim_Amount,Claim_Identifier,Claim,0.0,OK
6,Loss_Payment,Claim_Amount_Identifier,Claim_Payment,NaN,NOT CHECKED
7,Loss_Reserve,Claim_Amount_Identifier,Claim_Reserve,NaN,NOT CHECKED
8,Expense_Payment,Claim_Amount_Identifier,Claim_Payment,NaN,NOT CHECKED
9,Expense_Reserve,Claim_Amount_Identifier,Claim_Reserve,NaN,NOT CHECKED


In [10]:
# Đọc bảng Agreement
agreement_df = pd.read_csv(CSV_DIR / "Agreement.csv")

# Đọc dòng header gốc
with open(CSV_DIR / "Agreement.csv", encoding="utf-8-sig") as f:
    header = f.readline().strip().split(",")

print("Original columns:")
print(header)

print("\nColumns read by Pandas:")
print(agreement_df.columns.tolist())

print("\nAgreement data:")
display(agreement_df)

print("Has Agreement_Identifier:", "Agreement_Identifier" in agreement_df.columns)

Original columns:
['Agreement_Type_Code', 'Agreement_Name', 'Agreement_Original_Inception_Date', 'Product_Identifier', 'Agreement_Type_Code']

Columns read by Pandas:
['Agreement_Type_Code', 'Agreement_Name', 'Agreement_Original_Inception_Date', 'Product_Identifier', 'Agreement_Type_Code.1']

Agreement data:


,Agreement_Type_Code,Agreement_Name,Agreement_Original_Inception_Date,Product_Identifier,Agreement_Type_Code.1
0,1,Policy 31003000336,NaN,NaN,NaN
1,2,Policy 31003000337,NaN,NaN,NaN


Has Agreement_Identifier: False


In [11]:
from rdflib import Graph, RDF, OWL

# 1. Đọc OWL Ontology
ontology = Graph()
ontology.parse(ONTOLOGY_FILE, format="turtle")

# 2. Đếm các thành phần
classes = set(ontology.subjects(RDF.type, OWL.Class))
object_properties = set(ontology.subjects(RDF.type, OWL.ObjectProperty))
data_properties = set(ontology.subjects(RDF.type, OWL.DatatypeProperty))

# 3. Hiển thị kết quả
print("Total RDF Triples:", len(ontology))
print("Classes:", len(classes))
print("Object Properties:", len(object_properties))
print("Datatype Properties:", len(data_properties))

Total RDF Triples: 164
Classes: 11
Object Properties: 10
Datatype Properties: 18


In [12]:
from rdflib import Namespace

# 1. Đọc file R2RML Mapping
mapping = Graph()
mapping.parse(MAPPING_FILE, format="turtle")

# 2. Khai báo namespace R2RML
RR = Namespace("http://www.w3.org/ns/r2rml#")

# 3. Tìm các thành phần mapping
triple_maps = set(mapping.subjects(RDF.type, RR.TriplesMap))
logical_tables = set(mapping.objects(None, RR.logicalTable))
subject_maps = set(mapping.objects(None, RR.subjectMap))
predicate_object_maps = set(mapping.objects(None, RR.predicateObjectMap))

# 4. Hiển thị kết quả
print("Total RDF Triples:", len(mapping))
print("Triples Maps:", len(triple_maps))
print("Logical Tables:", len(logical_tables))
print("Subject Maps:", len(subject_maps))
print("Predicate-Object Maps:", len(predicate_object_maps))

Total RDF Triples: 561
Triples Maps: 62
Logical Tables: 62
Subject Maps: 62
Predicate-Object Maps: 50


In [13]:
# 1. Tìm quy tắc mapping cho Claim Number
example = next(
    tm for tm in triple_maps
    if "TripleMap_ClaimNumber" in str(tm)
)

# 2. Lấy các thành phần của quy tắc
logical_table = mapping.value(example, RR.logicalTable)
subject_map = mapping.value(example, RR.subjectMap)
predicate_object_map = mapping.value(example, RR.predicateObjectMap)
object_map = mapping.value(predicate_object_map, RR.objectMap)

# 3. Hiển thị
print("SQL Table:", mapping.value(logical_table, RR.tableName))
print("Subject Template:", mapping.value(subject_map, RR.template))
print("Predicate:", mapping.value(predicate_object_map, RR.predicate))
print("SQL Column:", mapping.value(object_map, RR.column))

SQL Table: myinsurancecompany.omg-pc-database.claim
Subject Template: https://myinsurancecompany.linked.data.world/d/omg-pc-database/Claim-{Claim_Identifier}
Predicate: http://data.world/schema/insurance/claimNumber
SQL Column: company_claim_number


In [14]:
from rdflib import Namespace

# 1. Đọc file benchmark
benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

# 2. Khai báo namespace
QA = Namespace("http://models.data.world/benchmarks/QandA#")

# 3. Tìm tất cả câu hỏi
inquiries = list(benchmark.subjects(RDF.type, QA.Inquiry))

# 4. Lấy nội dung từng câu hỏi
questions = []

for inquiry in inquiries:
    question = benchmark.value(inquiry, QA.prompt)

    if question is not None:
        questions.append(str(question))

# 5. Hiển thị kết quả
print("Total inquiries:", len(inquiries))
print("Total questions:", len(questions))
print("Unique questions:", len(set(questions)))

print("\nFirst 5 questions:")
for question in questions[:5]:
    print("-", question)

Total inquiries: 44
Total questions: 44
Unique questions: 44

First 5 questions:
- What is the average loss of each policy by policy number and number of claims where loss is the sum of loss payment, loss reserve, expense payment, expense reserve amounts?
- What are the total loss, which is the sum of loss payment, loss reserve, expense payment, expense reserve amount by catastrophe and policy number?
- Return all the policies and their policy holder by id
- How many claims have been placed by policy number?
- What is the average time to settle a claim by policy number?


In [15]:
# Namespace chứa các loại truy vấn
DWT = Namespace("https://templates.data.world/")

reference_results = []

# Duyệt từng câu hỏi
for inquiry in inquiries:
    question = str(benchmark.value(inquiry, QA.prompt))

    # Lấy các truy vấn tham chiếu
    refs = list(benchmark.objects(inquiry, QA.expects))

    # Phân loại SQL và SPARQL
    sql_refs = [
        ref for ref in refs
        if (ref, RDF.type, DWT.SqlQuery) in benchmark
    ]

    sparql_refs = [
        ref for ref in refs
        if (ref, RDF.type, DWT.SparqlQuery) in benchmark
    ]

    reference_results.append({
        "Question": question,
        "SQL": len(sql_refs),
        "SPARQL": len(sparql_refs),
        "Total": len(refs)
    })

# Tạo bảng kết quả
reference_df = pd.DataFrame(reference_results)

print("Total questions:", len(reference_df))
print("SQL references:", reference_df["SQL"].sum())
print("SPARQL references:", reference_df["SPARQL"].sum())

# Hiển thị các câu hỏi có số reference khác thông thường
print("\nQuestions requiring review:")

display(reference_df[
    (reference_df["SQL"] != 1) |
    (reference_df["SPARQL"] != 1)
])

Total questions: 44
SQL references: 45
SPARQL references: 44

Questions requiring review:


,Question,SQL,SPARQL,Total
38,Return agents and the policies they have sold ...,2,1,3


In [16]:
# 1. Lấy câu hỏi có 2 SQL Reference
question = reference_df[reference_df["SQL"] == 2].iloc[0]["Question"]

print("Question:")
print(question)

# 2. Tìm đối tượng Inquiry tương ứng
inquiry = next(
    i for i in inquiries
    if str(benchmark.value(i, QA.prompt)) == question
)

# 3. Lấy tất cả truy vấn tham chiếu
refs = list(benchmark.objects(inquiry, QA.expects))

# 4. Hiển thị từng truy vấn
for ref in refs:
    if (ref, RDF.type, DWT.SqlQuery) in benchmark:
        language = "SQL"
    else:
        language = "SPARQL"

    query = benchmark.value(ref, QA.queryText)

    print("\n---", language, "---")
    print(query)

Question:
Return agents and the policies they have sold that have had a claim and the corresponding loss payment amount by agent id, policy number and claim number

--- SQL ---
SELECT
    apr2.party_identifier as AgentID,
#    apr1.party_identifier as PolicyHolderID, 
    policy_number,
 #   policy_amount as premium, 
 #   catastrophe_name,
    company_claim_number,
    ca_lp.claim_amount as Loss_Payment_Amount
FROM
    Claim
    inner join catastrophe on claim.catastrophe_identifier = catastrophe.catastrophe_identifier
    inner join claim_amount ca_lp on claim.claim_identifier = ca_lp.claim_identifier
    inner JOIN loss_payment ON ca_lp.claim_amount_identifier = loss_payment.claim_amount_identifier
    inner join claim_amount ca_lr on claim.claim_identifier = ca_lr.claim_identifier
    inner JOIN loss_reserve ON ca_lr.claim_amount_identifier = loss_reserve.claim_amount_identifier
    inner join claim_amount ca_ep on claim.claim_identifier = ca_ep.claim_identifier
    inner JOIN expe